In [ ]:
!nvidia-smi


Wed Nov  4 14:31:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%pip install -q -U transformers accelerate

In [ ]:
import torch
import transformers

print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Transformers version: 5.22.1
GPU available: True
GPU name: Tesla T4


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model loaded:", model_id)
print("Model device:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Model device: cuda:0


In [ ]:
context = """
[P001] Per attivare la forma diretta, l’Assicurato deve chiedere,
con un preavviso di almeno 2 giorni lavorativi, un’autorizzazione
a MyAssistance per il pagamento diretto da parte della Compagnia.
Il preavviso non è necessario per i soli casi di urgenza medica
certificata.
"""

question = "Quanto preavviso serve per attivare la forma diretta?"

messages = [
    {
        "role": "system",
        "content": (
            "Rispondi in italiano usando solo le evidenze fornite. "
            "Conserva condizioni ed eccezioni. "
            "Cita il passaggio usando il suo identificativo, ad esempio [P001]. "
            "Se le informazioni non bastano, dichiaralo."
        ),
    },
    {
        "role": "user",
        "content": f"Evidenze:\n{context}\n\nDomanda:\n{question}",
    },
]

# Format the conversation as this model expects.
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Convert text into tokens and move them to the GPU.
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate an answer without calculating training gradients.
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Display only the newly generated answer.
answer_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

print(answer)

Per attivare la forma diretta, l'Assicurato deve chiedere con un preavviso di almeno 2 giorni lavorativi, come indicato nel passaggio [P001].


In [ ]:
# Keep the same evidence and system instruction.
# Replace only the question.
messages[1]["content"] = (
    f"Evidenze:\n{context}\n\n"
    "Domanda:\n"
    "In caso di urgenza medica certificata, "
    "devo comunque dare due giorni lavorativi di preavviso?"
)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

answer_tokens = outputs[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(answer_tokens, skip_special_tokens=True))

No, in caso di urgenza medica certificata, non è necessario dare un preavviso di almeno 2 giorni lavorativi per attivare la forma diretta. Questo tipo di richiesta può essere effettuata direttamente con l'autorizzazione a MyAssistance senza alcun preavviso. [P001]


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Zurich_Salute_SLM_Dataset_v1_DRAFT.zip to Zurich_Salute_SLM_Dataset_v1_DRAFT.zip


In [ ]:
import json
import zipfile
from pathlib import Path

zip_name = "Zurich_Salute_SLM_Dataset_v1_DRAFT.zip"
data_dir = Path("/content/navisio_data")

with zipfile.ZipFile(zip_name, "r") as archive:
    archive.extractall(data_dir)

# Locate the training file inside the extracted folder.
train_path = next(data_dir.rglob("train_draft.jsonl"))

with train_path.open(encoding="utf-8") as file:
    train_examples = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("Training candidates:", len(train_examples))

example = train_examples[0]

print("\nQUESTION:")
print(example["question_it"])

print("\nREFERENCE ANSWER:")
print(example["target_answer_it"])

print("\nREVIEW STATUS:")
print(example["review_status"])

Training candidates: 81

QUESTION:
La copertura sanitaria mondiale consente di assicurare anche chi risiede all'estero?

REFERENCE ANSWER:
No. L'assicurazione è valida per i soli assicurati residenti nel Territorio Italiano. Spese sanitarie, Alta Diagnostica, Visite specialistiche e Diaria da Ricovero valgono in tutto il mondo; ciò non elimina il requisito di residenza. [ZS_P014]

REVIEW STATUS:
draft_pending_independent_review


In [ ]:
example = train_examples[0]

# Build the evidence from this example's source passages.
evidence = "\n\n".join(
    f"[{passage['passage_id']}]\n{passage['text']}"
    for passage in example["context"]
)

# Only instructions, evidence and question go to Qwen.
messages = [
    {
        "role": "system",
        "content": (
            "Rispondi in italiano usando solo le evidenze fornite. "
            "Conserva condizioni ed eccezioni e cita i passaggi. "
            "Se le informazioni non bastano, dichiaralo."
        ),
    },
    {
        "role": "user",
        "content": (
            f"Evidenze:\n{evidence}\n\n"
            f"Domanda:\n{example['question_it']}"
        ),
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

answer_tokens = outputs[0, inputs["input_ids"].shape[1]:]
model_answer = tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True,
)

print("MODEL ANSWER:\n", model_answer)
print("\nDRAFT REFERENCE ANSWER:\n", example["target_answer_it"])

MODEL ANSWER:
 No, la copertura sanitaria mondiale non consente di assicurare chi risiede all'estero. L'articolo specificamente indica che "Spese sanitarie, Alta Diagnostica, Visite specialistiche e Diaria da Ricovero" sono valide in tutto il mondo, ma "Esami di routine" sono valide solo nel Territorio Italiano nei centri convenzionati con MyAssistance.

DRAFT REFERENCE ANSWER:
 No. L'assicurazione è valida per i soli assicurati residenti nel Territorio Italiano. Spese sanitarie, Alta Diagnostica, Visite specialistiche e Diaria da Ricovero valgono in tutto il mondo; ciò non elimina il requisito di residenza. [ZS_P014]


In [ ]:
baseline_results = [{
    "example_id": example["id"],
    "question": example["question_it"],
    "model_answer": model_answer,
    "reference_answer": example["target_answer_it"],
    "assessment": {
        "correct_conclusion": True,
        "residence_requirement_explicit": False,
        "citation_present": False,
        "notes": "Correct conclusion; incomplete supporting explanation."
    }
}]

print("Saved baseline results:", len(baseline_results))

Saved baseline results: 1


In [ ]:
def answer_example(example):
    evidence = "\n\n".join(
        f"[{p['passage_id']}]\n{p['text']}"
        for p in example["context"]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "Rispondi in italiano usando solo le evidenze fornite. "
                "Conserva condizioni ed eccezioni e cita i passaggi. "
                "Se le informazioni non bastano, dichiaralo."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Evidenze:\n{evidence}\n\n"
                f"Domanda:\n{example['question_it']}"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
example = next(
    item for item in train_examples
    if item["task"] == "clarification"
)

model_answer = answer_example(example)

print("QUESTION:\n", example["question_it"])
print("\nMODEL ANSWER:\n", model_answer)
print("\nDRAFT REFERENCE:\n", example["target_answer_it"])

QUESTION:
 Ho acquistato Zurich Salute: ho sicuramente attiva la garanzia Spese sanitarie?

MODEL ANSWER:
 Per rispondere alle domande sulle garanzie Spese sanitarie di Zurich Salute, dovremmo analizzare attentamente il testo fornito:

1. **Garanzia Spese sanitarie**: La garanzia Spese sanitarie è opera di polizza e viene applicata solo quando viene richiamata.

2. **Oggetto dell'assicurazione**:
   - Le spese sostenute e documentate per ricovero con o senza intervento chirurgico.
   - Day Surgery.
   - Intervento chirurgico ambulatoriale.
   - Esami, accertamenti diagnostici e visite mediche specialistiche.
   - Onorari dell'équipe medica.
   - Diritti di sala operatoria e materiali di intervento.
   - Rette di degenza, prestazioni mediche, consulenti, cure mediche, assistenza infermieristica, trattamenti fisioterapici e rieducativi, medicamentosi, esami ed accertamenti diagnostici.
   - Vitto e pernottamento dell'accompagnatore.
   - Spese relative a

DRAFT REFERENCE:
 Non posso conf

In [ ]:
required = [
    "model",
    "tokenizer",
    "train_examples",
    "answer_example",
    "baseline_results",
]

for name in required:
    status = "available" if name in globals() else "missing"
    print(f"{name}: {status}")

model: available
tokenizer: available
train_examples: available
answer_example: available
baseline_results: available


In [ ]:
import json
from pathlib import Path

validation_path = next(
    Path("/content/navisio_data").rglob("validation_draft.jsonl")
)

with validation_path.open(encoding="utf-8") as file:
    validation_examples = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("Validation examples:", len(validation_examples))

Validation examples: 22


In [ ]:
from datetime import datetime, timezone
from google.colab import files

# A separate list preserves your earlier practice results.
validation_baseline = []

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_path = Path(f"/content/validation_baseline_{run_id}.jsonl")

model.eval()

with output_path.open("w", encoding="utf-8") as file:
    for i, example in enumerate(validation_examples, start=1):
        answer = answer_example(example)

        result = {
            "example_id": example["id"],
            "model_id": model_id,
            "model_revision": getattr(model.config, "_commit_hash", None),
            "split": "validation",
            "question": example["question_it"],
            "context": example["context"],
            "model_answer": answer,
            "draft_reference_answer": example["target_answer_it"],
            "reference_review_status": example["review_status"],
            "generation_settings": {
                "max_new_tokens": 250,
                "do_sample": False,
            },
            "assessment": None,
        }

        validation_baseline.append(result)

        # Save each answer immediately as the loop progresses.
        file.write(json.dumps(result, ensure_ascii=False) + "\n")
        file.flush()

        print(f"Completed {i}/{len(validation_examples)}")

print("\nSaved:", output_path.name)
files.download(str(output_path))

Completed 1/22
Completed 2/22
Completed 3/22
Completed 4/22
Completed 5/22
Completed 6/22
Completed 7/22
Completed 8/22
Completed 9/22
Completed 10/22
Completed 11/22
Completed 12/22
Completed 13/22
Completed 14/22
Completed 15/22
Completed 16/22
Completed 17/22
Completed 18/22
Completed 19/22
Completed 20/22
Completed 21/22
Completed 22/22

Saved: validation_baseline_20261104T144901Z.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import csv
from google.colab import files

csv_path = output_path.with_suffix(".csv")

with csv_path.open("w", encoding="utf-8-sig", newline="") as file:
    columns = [
        "example_id",
        "question",
        "model_answer",
        "draft_reference_answer",
        "correctness",
        "completeness",
        "citation_support",
        "review_notes",
    ]

    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()

    for result in validation_baseline:
        writer.writerow({
            column: result.get(column, "")
            for column in columns
        })

files.download(str(csv_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>